# IS 327 - Homework 3
# Due date: Monday Apr 14, 10:59 am (right before class)

Everything you need to complete is marked by the word "TODO". In general, every individual part of "TODO" can be completed by a few lines of code (using the sklearn functions we provided in the instructions). We are not asking for any complicated manual implementation.

I discovered several issues with the HW2. Therefore, we will continue using the student success dataset. Moreover, we will use support vector machines in this homework. This class of algorithms is sensitive to scaling, making it a good example for you to practice scaling. Also, you will practice finding ways to improve your ML models. The accuracy scores you got before using the decision tree model were between 0.70 and 0.90. Let's see whether we can improve the models.

## 1. Dataset: Predict Students's status

Dataset: https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success

You can read about what each column is about on the same page.

Download this dataset and unzip it.

Place "data.csv" in the same directory as your jupyter notebook.

In [2]:
# import data
import pandas as pd
df = pd.read_csv("data (1).csv", sep=";")
# remove the enrolled class
df = df[df['Target'] != 'Enrolled']

In [3]:
# we reset the index so the indices won't have gaps
df = df.reset_index()
df.tail()

,index,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
3625,4419,1,1,6,9773,1,1,125.0,1,1,...,0,6,8,5,12.666667,0,15.5,2.8,-4.06,Graduate
3626,4420,1,1,2,9773,1,1,120.0,105,1,...,0,6,6,2,11.000000,0,11.1,0.6,2.02,Dropout
3627,4421,1,1,1,9500,1,1,154.0,1,37,...,0,8,9,1,13.500000,0,13.9,-0.3,0.79,Dropout
3628,4422,1,1,1,9147,1,1,180.0,1,37,...,0,5,6,5,12.000000,0,9.4,-0.8,-3.12,Graduate
3629,4423,1,10,1,9773,1,1,152.0,22,38,...,0,6,6,6,13.000000,0,12.7,3.7,-1.70,Graduate


## 2. Examine features from a different perspective

In the last two homework assignments, I mainly prompted you to think about the meaning of the features. This time, we need to think about the numbers.

Many of you were surprised that features such as parents' qualifications and occupations did not have much effect on prediction, but intutively they should. Let's see why.

**TODO 1** (5 pt)

Go to the dataset website https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success
And examine how "Nacionality" is encoded ("Nationality") in the "description" column of the Variable Table. Are the numerical values meaningful? Hint: Does being German half way between being Portuguese and being Spanish?

Answer: The numbers are not meaningful, they are as meaningful numerically as much as a categorical label for German would have. Since as the hint indicates, finding the average of these numerical values are not meaningful, therefore despite having numerical values, this data is still categorical.  

**TODO 2** (5 pt)

Now examine the following four features: Mother's qualification, Father's qualification, Mother's occupation, and Father's occupation.
Do you have an answer as to why, if we use each feature as is, they are not useful in predicting students' outcomes? 

Answer: For the same reason the numbers associated with nationalities are not truly meaningful, the numbers associated with occupations and qualifications are not also. For instance, when looking at numbers 0-2, "0 - Student 1 - Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers 2 - Specialists in Intellectual and Scientific Activities ", these numbers close to each other represent wildly different occupations. If we look at the last few occupations, we see "92 - Unskilled workers in agriculture, animal production, fisheries and forestry 193 - Unskilled workers in extractive industry, construction, manufacturing and transport 194 - Meal preparation assistants". These few are also quite different from each other. Are these numbers meant to represent level of education required to have the occupation? Are they meant to represent something else if the numbers are actually used for calculations? Either way, it is not an effective metric to be used for more advanced calculations. Also, it would be questionable from an ethical perspective to somewhat "rank" occupations in such a way if that was the intention. The values derrived from making calculations with these numbers would not have meaning or an ethical meaning at least. 


## 3. Properly use parents' qualifications and occupations in your model

We will do one-hot encoding to transform these features into multiple ones. I will demo how I did it with the feature "Mother's qualification." You will need to finish the remaining yourself. 

In [4]:
# use \' instead of '
m_qual = df['Mother\'s qualification']

# examine how many levels this variable has
print(f'No. of Levels: {len(m_qual.unique())}')

No. of Levels: 29


In [5]:
m_qual.unique()

array([19,  1, 37, 38,  3, 42,  2, 34,  4, 12, 40,  9,  5, 39, 11, 41, 30,
       14, 35, 36,  6, 10, 29, 18, 43, 22, 27, 26, 44], dtype=int64)

There are 29 levels, and we will use one-hot-encoding to expand the one column into 29 columns.

**TODO 3** (5 pt)

Ask LLMs about one-hot encoding. In particular, when is it crucial to do one-hot encoding?

Answer: "One‑hot encoding is crucial when you have categorical features that are nominal—that is, when the numerical labels do not carry any intrinsic order or magnitude. Directly using the numerical labels can mislead many machine learning algorithms into assuming a relationship (or distance) between categories where none exists. For example, an SVM might inappropriately interpret a numerical increase as a progression or weighting, which is not meaningful for categorical data. By converting these features into multiple binary columns (each representing the presence or absence of a specific category), one‑hot encoding prevents the algorithm from inferring non-existent numeric relationships. This is particularly important for models that base decisions on distance or dot products, such as SVMs, logistic regression, or K‑nearest neighbors." ChatGPT Output. It states that it is most important to use one-hot encoding in this exact application: when numbers themselves are not meaningful and represent categorical data. 

In [6]:
# One-hot encoding
# set dtype so it shows zeros and 1s
# you need to set the columns argument so that the actual values are recorded in the column header
m_qual_encoded = pd.get_dummies(m_qual, dtype=float, prefix = 'm_qual', columns=['Mother\'s qualification'])

# Check the shape of the new dataframe
m_qual_encoded.shape

(3630, 29)

In [7]:
# check how the dataframe look like
m_qual_encoded

,m_qual_1,m_qual_2,m_qual_3,m_qual_4,m_qual_5,m_qual_6,m_qual_9,m_qual_10,m_qual_11,m_qual_12,...,m_qual_35,m_qual_36,m_qual_37,m_qual_38,m_qual_39,m_qual_40,m_qual_41,m_qual_42,m_qual_43,m_qual_44
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3626,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3627,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3628,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


⚠️ This is what we call a sparse matrix, as most values are zeros. Sparse data are often regarded as a waste of storage space. They can also affect the training and final model performance. 

In [8]:
# the collumns match the values in appeared in this feature
# notice they are not continous. There are a few gaps. 
val_in_m_qual = list(m_qual.unique())
val_in_m_qual.sort()
print(val_in_m_qual)

[1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 14, 18, 19, 22, 26, 27, 29, 30, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]


**TODO 4** (5 pt)

Ask LLMs about what the effect of sparse data on machine learning algorithms? You can ask in general or specifically for SVMs. Tell me what you found.

Answer: ChatGPT discussed potential issues with having sparse data after having done one hot encoding. These included increased dimentionality, computational inefficiency, reduced discriminative power, and the kernel impact in SVMs. All of this is to say that as a result of the restructuring of the numerical data, there will be much more meaningless data required to represent the same information. This leads to the potential for memory problems, difficulty with the algorithm being able to find patterns that are relevant to the problem, and non-linear kernels could struggle without being configured properly.

In [11]:
# TODO 5 (5 pt)
# perform the same one-hot encoding on feature 'Father\'s qualification'
# the resulting variables should be: f_qual and f_qual_encoded
# you won't be able to run some later code witout observing this naming convention

# code here
f_qual = df['Father\'s qualification']

# examine how many levels this variable has
print(f'No. of Levels: {len(f_qual.unique())}')

f_qual.unique()

f_qual_encoded = pd.get_dummies(f_qual, dtype=float, prefix = 'f_qual', columns=['Father\'s qualification'])

# Check the shape of the new dataframe
f_qual_encoded.shape

No. of Levels: 34


(3630, 34)

In [13]:
# TODO 6 (5 pt)
# perform the same one-hot encoding on feature 'Mother\'s occupation'
# the resulting variables should be: m_occup and m_occup_encoded
# you won't be able to run some later code witout observing this naming convention

# code here
m_occ = df['Mother\'s occupation']

# examine how many levels this variable has
print(f'No. of Levels: {len(m_occ.unique())}')

m_occ.unique()

m_occ_encoded = pd.get_dummies(m_occ, dtype=float, prefix = 'm_occ', columns=['Mother\'s occupation'])

# Check the shape of the new dataframe
m_occ_encoded.shape

No. of Levels: 29


(3630, 29)

In [12]:
# TODO 7 (5 pt)
# perform the same one-hot encoding on feature 'Father\'s occupation'
# the resulting variables should be: f_occup and f_occup_encoded
# you won't be able to run some later code witout observing this naming convention

# code here
f_occ = df['Father\'s occupation']

# examine how many levels this variable has
print(f'No. of Levels: {len(f_occ.unique())}')

f_occ.unique()

f_occ_encoded = pd.get_dummies(f_occ, dtype=float, prefix = 'f_occ', columns=['Father\'s occupation'])

# Check the shape of the new dataframe
f_occ_encoded.shape

No. of Levels: 42


(3630, 42)

## 3. Build a SVM classifier WITHOUT parents' qualifications and occupations

We want to try to build a SVM classifier without parents' qualifications and occupations. 

I selecte a few features for you. I did not include the controversial feature "Tuition fees up to date" known to increase model performance. 

In [14]:
# features to be included
feature_names = [
    'Daytime/evening attendance\t',
    'Previous qualification (grade)',
    'Admission grade',
    'Educational special needs',
    'Debtor',
    'Scholarship holder',
    'Age at enrollment',
    'International',
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 2nd sem (without evaluations)']

In [15]:
# create our first dataset for model building
# and thus will make our models less useful for meaningful prediction
X_1 = df[feature_names]
y_1 = df['Target']

In [16]:
# train/test split
from sklearn.model_selection import train_test_split
X_1_train, X_1_test, y_1_train, y_1_test = train_test_split(X_1, y_1, test_size=0.2, random_state=42)

In [17]:
# TODO 8 (5 pt)
# train a SVC classifier using the X_1_train, X_1_test, y_1_train, y_1_test
# name the classifier svc_1
# MAKE SURE you use the following parameters
# kernel = 'linear'
# gamma = 'scale'
# C = 10
# https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
from sklearn.svm import SVC

# code here
svc_1 = SVC(kernel='linear', gamma='scale', C=10, random_state=42)
svc_1.fit(X_1_train, y_1_train)

SVC(C=10, kernel='linear', random_state=42)

In [18]:
# TODO 9 (5 pt)
# use classification_report to print the metrics
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from sklearn.metrics import classification_report

# code here
y_1_pred = svc_1.predict(X_1_test)

report_svc1 = classification_report(y_1_test, y_1_pred)
print(report_svc1)

              precision    recall  f1-score   support

     Dropout       0.91      0.80      0.85       277
    Graduate       0.89      0.95      0.92       449

    accuracy                           0.90       726
   macro avg       0.90      0.88      0.89       726
weighted avg       0.90      0.90      0.89       726



**TODO 10** (5 pt)

Examine the classification report. How does the model compare to your previous DT model (checking your HW2)?

Answer: This is a significant improvement from my last model when using the decision tree. The recall for the dropout category is significantly better, but it is still the area that could use the greatest improvement compared to the other scores. Perhaps a next step could be adjusting the kernel fuctions or adjusting the C parameter. 

## 4. Build a SVM classifier WITH parents' qualifications and occupations

Now we will bring the transformed features into our training dataset.

⚠️I am a bit lazy here by assuming the order of the rows hasn't changed, and I will horizontally stack all rows. In reality, in order to be more defensive, you should assign unique IDs to the instances and use SQL to merge them. 

In [20]:
X_2 = pd.concat([df[feature_names], m_qual_encoded, f_qual_encoded, m_occ_encoded, f_occ_encoded], axis=1)
y_2 = df['Target']

In [28]:
# see how X_2 look like
X_2.head()

,Daytime/evening attendance\t,Previous qualification (grade),Admission grade,Educational special needs,Debtor,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),...,f_occ_172,f_occ_174,f_occ_175,f_occ_181,f_occ_182,f_occ_183,f_occ_192,f_occ_193,f_occ_194,f_occ_195
0,1,122.0,127.3,0,0,0,20,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,160.0,142.5,0,0,0,19,0,0,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,122.0,124.8,0,0,0,19,0,0,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,122.0,119.6,0,0,0,20,0,0,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,100.0,141.5,0,0,0,45,0,0,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**TODO 11** (5 pt)

What have been changed from X_2 to X_3

Answer: X_3 is a scaled version of X_2 with a minmaxscaler. This is done to ensure no feature has a disproportionate impact on the SVM's distance calculations, which is important for sensitive models to feature scales. So it has the same 154 columns, but rescaled between 0 and 1. To my understanding, X_3 is just X_2 normalized and stored as a new df in the notebook memory. 

In [29]:
# another train/test split
from sklearn.model_selection import train_test_split
X_2_train, X_2_test, y_2_train, y_2_test = train_test_split(X_2, y_2, test_size=0.2, random_state=42)

In [30]:
# TODO 12 (5 pt)
# train a SVC classifier using the X_2_train, X_2_test, y_2_train, y_2_test
# NAME the new classifer svc_2
# MAKE SURE you use the following parameters
# kernel = 'linear'
# gamma = 'scale'
# C = 10
# https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
from sklearn.svm import SVC

# code here
svc_2 = SVC(kernel='linear', gamma='scale', C=10, random_state=42)
svc_2.fit(X_2_train, y_2_train)

SVC(C=10, kernel='linear', random_state=42)

In [31]:
# TODO 13 (5 pt)
# use classification_report to print the metrics
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from sklearn.metrics import classification_report

# code here
y_2_pred = svc_2.predict(X_2_test)

print(classification_report(y_2_test, y_2_pred))

              precision    recall  f1-score   support

     Dropout       0.90      0.83      0.86       277
    Graduate       0.90      0.94      0.92       449

    accuracy                           0.90       726
   macro avg       0.90      0.89      0.89       726
weighted avg       0.90      0.90      0.90       726



**TODO 14** (5 pt)

Examine the classification report. How did the new model show improvement?

Answer: A lot of the values remained the same. Some did actually go down, but in return, others went up by 0.01, and recall for dropout went up 0.03, which is great. I would say this is a preferable model to the previous one. 

## 4. Build a SVM classifier WITH parents' qualifications and occupations AND scaling

Most of features in "feature_names" have varied ranges, for instance, age (roughly between 20 to 70), admission grade (roughly between 100 to 200), and curricular units (0 to 20). On the contrary, all the transformed parents' qualifications and occupations features have only two values: 0 and 1. SVMs rely on calculating distances between data points to determine their relationships and classify them. When features have different scales (e.g., one feature ranges from 0 to 1000, while another ranges from 0 to 1), the feature with the larger scale will dominate the distance calculations, potentially masking the importance of other features.

In [32]:
df[['Admission grade', 'Age at enrollment', 'Curricular units 1st sem (credited)']].describe()

,Admission grade,Age at enrollment,Curricular units 1st sem (credited)
count,3630.000000,3630.000000,3630.000000
mean,127.293939,23.461157,0.754270
std,14.611295,7.827994,2.477277
min,95.000000,17.000000,0.000000
25%,118.000000,19.000000,0.000000
50%,126.500000,20.000000,0.000000
75%,135.100000,25.000000,0.000000
max,190.000000,70.000000,20.000000


This is a good place to try to use scaling. We will use `MinMaxScaler` from sklearn. I wrote the code for you. 

In [33]:
# TODO 15 (5 pt)
# use MinMaxScaler to scale your X_2
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
# transform X_2 to X_3
from sklearn.preprocessing import MinMaxScaler

# code here
scaler = MinMaxScaler()

X_3_array = scaler.fit_transform(X_2)

X_3 = pd.DataFrame(X_3_array, columns=X_2.columns)

print(X_3.head())

   Daytime/evening attendance\t  Previous qualification (grade)  \
0                           1.0                        0.284211   
1                           1.0                        0.684211   
2                           1.0                        0.284211   
3                           1.0                        0.284211   
4                           0.0                        0.052632   

   Admission grade  Educational special needs  Debtor  Scholarship holder  \
0         0.340000                        0.0     0.0                 0.0   
1         0.500000                        0.0     0.0                 0.0   
2         0.313684                        0.0     0.0                 0.0   
3         0.258947                        0.0     0.0                 0.0   
4         0.489474                        0.0     0.0                 0.0   

   Age at enrollment  International  Curricular units 1st sem (credited)  \
0           0.056604            0.0                       

In [37]:
# examine the X_3 and compare it to X_2
X_3.head()

,Daytime/evening attendance\t,Previous qualification (grade),Admission grade,Educational special needs,Debtor,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),...,f_occ_172,f_occ_174,f_occ_175,f_occ_181,f_occ_182,f_occ_183,f_occ_192,f_occ_193,f_occ_194,f_occ_195
0,1.0,0.284211,0.340000,0.0,0.0,0.0,0.056604,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.684211,0.500000,0.0,0.0,0.0,0.037736,0.0,0.0,0.230769,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.284211,0.313684,0.0,0.0,0.0,0.037736,0.0,0.0,0.230769,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.284211,0.258947,0.0,0.0,0.0,0.056604,0.0,0.0,0.230769,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.052632,0.489474,0.0,0.0,0.0,0.528302,0.0,0.0,0.230769,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
# Prepare the data again for model training
from sklearn.model_selection import train_test_split
X_3_train, X_3_test, y_3_train, y_3_test = train_test_split(X_3, y_2, test_size=0.2, random_state=42)

In [39]:
# TODO 16 (5 pt)
# Use the new X_3_train, y_3_train train a SVC (svc_3) classifier with the same parameters
# kernel = 'linear'
# gamma = 'scale'
# C = 10

from sklearn.svm import SVC

# code here
svc_3 = SVC(kernel='linear', gamma='scale', C=10, random_state=42)
svc_3.fit(X_3_train, y_3_train)

SVC(C=10, kernel='linear', random_state=42)

In [40]:
# TODO 17 (5 pt)
# use classification_report to print the metrics
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from sklearn.metrics import classification_report

# code here
y_3_pred = svc_3.predict(X_3_test)
print(classification_report(y_3_test, y_3_pred))

              precision    recall  f1-score   support

     Dropout       0.90      0.83      0.86       277
    Graduate       0.90      0.94      0.92       449

    accuracy                           0.90       726
   macro avg       0.90      0.89      0.89       726
weighted avg       0.90      0.90      0.90       726



This is only for you to think about. Not a TODO.

Compare the classification report from svc_1, svc_2, and svc_3, what did you find? Just let you know that I did not see a significant improvement, so if you saw the same thing, please don't worry. After all, we tried to correct issues we found, and this is the most important thing. 

⚠️ SVMs are computational costly, so I don't want to burden people with cross-validations (short hand as x-val from now on). With x-val, you might find a hyperparameter combination that can improve the metrics. You are welcome to try if your computer can handle it. Futhermore, dimension reduction may help, which is something we can try in the future. 

You can use coefficient to approximate feature importance for SVMs with linear kernels. Let's find out what are the most indicative features for students' success. 

In [41]:
coef_list = svc_3.coef_[0].tolist()
feature_list = list(X_3.columns)
coef_dict = {x:y for (x,y) in zip(feature_list, coef_list)}

In [42]:
# display your notebook
coef_dict

{'Daytime/evening attendance\t': -0.20251286023858484,
 'Previous qualification (grade)': -0.22993491430986523,
 'Admission grade': 0.5536741730048789,
 'Educational special needs': 0.07180006451408616,
 'Debtor': -1.1104039381430653,
 'Scholarship holder': 0.7174617677011952,
 'Age at enrollment': -0.8047371088890145,
 'International': 0.26926856311365555,
 'Curricular units 1st sem (credited)': -1.1452055587681258,
 'Curricular units 1st sem (enrolled)': -5.39061763747344,
 'Curricular units 1st sem (evaluations)': 1.0411538384194614,
 'Curricular units 1st sem (approved)': 9.214822043529182,
 'Curricular units 1st sem (grade)': -0.49072556060787065,
 'Curricular units 1st sem (without evaluations)': -0.680558832937507,
 'Curricular units 2nd sem (credited)': -2.351652019889528,
 'Curricular units 2nd sem (enrolled)': -8.804689388751228,
 'Curricular units 2nd sem (evaluations)': -1.2732766854712168,
 'Curricular units 2nd sem (approved)': 12.065122968001788,
 'Curricular units 2nd s

**TODO 18** (5 pt)

Use the dictionary of coefficients to identify the most important feature (largest ABSOLUTE value). What is it? Did it ring any bell (e.g., you've seen it as an important feature from HW2)

Answer: Curricular units 2nd sem, with the coefficient of approx. 12.07 was the feature with the largest aboslute value. This value was definitely an important feature from HW2 and was featured in many of the selected features we looked at in the code. 

**TODO 19** (5 pt) 

Examine the following features from Parents' occupations and qualifications. They have relatively high coefficients. Do they make sense as predictors for students' success? Numbers correspond to the encoding in the variable table. For example, m_qual_1 corresponds to when the value in "Mother's qualification" is 1, meaning "Secondary Education." Ignore the signs (i.e., positive or negative). 

https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success

Feature 1. `f_qual_35`

The value you got: -1.89

Does the value make sense: The numerical coding is arbitrary, so while the model seems to indicate this has predictive power, it's difficult to determine if the category truly "ranks" higher meaningfully. 

Feature 2.  `f_qual_36`

The value you got: -0.035

Does the value make sense: The value is near zero, suggesting this feature does not have much predictive power. It does not discriminate between successful and unsuccessful outcomes very well. 

Feature 3. `m_occup_123`

The value you got: -1.13

Does the value make sense:thsi hsows moderate influence, but since this is also a one-hot encoded feature derrived from the numerical values that are not really meaningful, it is difficult to know if we have found a causal relationship/interpretation. 


## 5. Conclusions

I hope you learned something from this assignment. I had a good time making it. The last TODO is to write a short reflection passage about what you learned from the process. What can you use in the future with your own ML project?

Additionally, I want to comment on the difference between decision tree (DT) and SVM. First, DT will have a hard time making use of the parents' occupations and qualifications features as they are represented in the dataset because it is hard to say where you can place a boundary. Moreover, once we transform those four features into one-hot encoding, DT may also find trouble handling so many dimensions. SVMs are much better at handling high-dimensionality data. 

**TODO 20** (5 pt)

Answer: I learned about modes of preprocessing, particularly one-hot encoding categorical data and then subsequently scaling it with the MinMaXScaler. Both of these things are important for SVM performance since they insure equal contributions of all features to distance calculations as well as preserving categorical characteristics of data. My understanding of feature engineering and evaluating quality of models has been deepened for sure, and I will definitely keep these techniques in mind in future ML projects, particularly the final project. 

**References (if any)** here

Cite any resources you used in the notebook. If you used some code from others, please put their links here.


**Declare GenAI usage** here

Again, we **encourage** the use of GenAI. You only need to declare. 

I used it to help me write the code for many of the TODOs, guide some of my short answer questions, as well as parse through the long list to find my largest absolute values.
